# Solution: Download, Run, and Fine-Tune a Real Model

**Unit:** Training & Tradeoffs | **Companion reading:** Lessons 1-5  
**Suggested time:** 2-3 hours (Part 1-2 in session one; Part 3-4 in session two)

---

## What is this assignment trying to do?

You have used ChatGPT-style products. This assignment shows **what happens underneath** when a model becomes useful for a specific job.

In four steps you will:

1. **Download** a real pretrained model from the internet (you are not building one from scratch).
2. **Run** it and observe a raw **base model** - it often *continues* text instead of *following instructions*.
3. **Fine-tune** it on examples of one narrow task (starter task: casual to formal business English).
4. **Track** training with **Weights & Biases (W&B)** so you can compare experiments the way ML teams do.

**Big idea:** Pretraining teaches general language (Lesson 1). Fine-tuning nudges the same weights toward *your* task using a small dataset and a short training run. You are doing a miniature version of what product teams do before launch.

**Model:** `distilgpt2` (~82 million parameters). Small enough to fine-tune in minutes on Colab.

| Part | What you do | Why |
|------|-------------|-----|
| 1 | Load base model, generate text | See "before" behavior |
| 2 | Build 50+ input/output examples | Give the model teaching material |
| 3 | Train twice, log to W&B | Practice hyperparameters + experiment tracking |
| 4 | Compare before/after | Show whether fine-tuning helped |


**Instructor reference.**


---
## Key terms (read this first)

| Term | Plain-English meaning |
|------|------------------------|
| **Pretrained model** | Already trained on huge text. You download weights instead of training from zero. |
| **Base model** | Pretrained but not customized for your task. Strong at text completion. |
| **Fine-tuning** | Extra training on *your* examples to specialize behavior. |
| **Hugging Face** | Hub + Python library for models, tokenizers, and datasets. |
| **Tokenizer** | Converts text to numbers (tokens) the model can process. |
| **Prompt** | Text you provide before the model generates more text. |
| **Generate** | Model predicts upcoming tokens and returns new text. |
| **Loss** | Error score during training. **Lower is better.** |
| **Epoch** | One full pass through all training examples. |
| **Learning rate** | Size of each weight update step. |
| **Batch size** | Number of examples processed before one update. |
| **Weights & Biases (W&B)** | Free website that logs loss curves and settings for each training run. |
| **Checkpoint** | Saved model file after training (your fine-tuned model). |

**Analogy:** Pretraining = reading the whole library. Fine-tuning = practicing one interview format. W&B = lab notebook that records every practice session.


---
## Part 0: Setup

### Where to run

**Google Colab is recommended** (free notebook + optional GPU).

1. Upload this file to Colab.
2. **Runtime -> Change runtime type -> T4 GPU** before Part 3.
3. Run cells top to bottom.

### Libraries

| Library | Purpose |
|---------|---------|
| `transformers` | Load Hugging Face models; run training |
| `datasets` | Hold your training examples |
| `accelerate` | Required helper for `Trainer` |
| `wandb` | Log metrics to Weights & Biases |
| `torch` | PyTorch (deep learning engine) |

Run the next cell. In Colab, uncomment `pip install` if imports fail.


In [ ]:
# Colab: uncomment if imports fail
# !pip install -q transformers datasets accelerate wandb

import os
import random
import textwrap
import time

import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("Tip: enable Colab GPU before Part 3 for faster training.")

MODEL_NAME = "distilgpt2"
WANDB_PROJECT = "forge-llm-finetune"  # TODO: add your name


---
## Part 1: Download and Run a Pretrained Model

### What you are doing (no training yet)

1. **Download** weights from Hugging Face (~350 MB for distilgpt2).
2. **Load** a matching **tokenizer** (text <-> numbers).
3. **Generate** text: the model predicts likely next tokens.

`distilgpt2` learned **next-token prediction** on web text. It is **not** instruction-tuned like ChatGPT. When you write "Write a haiku," it may repeat the question - that is expected.

**Save these outputs** - Part 4 compares base vs fine-tuned on similar prompts.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
base_model.to(device)
base_model.eval()

print(f"Loaded {MODEL_NAME}")
print(f"Parameters: {base_model.num_parameters() / 1e6:.1f} million")


In [ ]:
def generate_text(model, prompt, max_new_tokens=80, temperature=0.8):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


BASELINE_PROMPTS = [
    "Once upon a time",
    "Question: What is the capital of France?\nAnswer:",
    "Write a haiku about rain.",
    "Translate to pirate: Hello friend\n",
]

print("=== Base model outputs - SAVE for Part 4 ===\n")
for p in BASELINE_PROMPTS:
    result = generate_text(base_model, p)
    print("PROMPT:", repr(p))
    print(textwrap.fill(result, width=100))
    print("-" * 60)


### Part 1 reflection

**Question:** Does the model follow instructions, or mostly continue the prompt? Why?

**Hint:** Lesson 1 - base models optimize next-token prediction, not helpful-assistant behavior. ChatGPT added later fine-tuning and RLHF; distilgpt2 did not.

Write your answer in the next cell.


In [ ]:
part1_reflection = "Base model continues text; not instruction-tuned like ChatGPT."


---
## Part 2: Prepare a Fine-Tuning Dataset

### What is fine-tuning data?

Each training row has:
- **Input** - what a user might type
- **Output** - the correct answer you want

We format them as one string:

```
### Input:
hey can u send the report

### Output:
Could you please send the report at your earliest convenience.
```

The model reads the full string and learns patterns. After enough examples, it learns: after `### Output:` for casual inputs, produce formal text.

### TODO 1

Starter task: **casual -> formal business English**.

You get **25 examples below**. Add **at least 25 more** (50 minimum). Or replace everything with a different task and update `TASK_NAME`.


In [ ]:
TASK_NAME = 'casual_to_formal'
TRAINING_PAIRS = [
    ("hey can u send the report", "Could you please send the report at your earliest convenience."),
    ("sry im late", "I apologize for my tardiness."),
    ("thx for the help", "Thank you for your assistance."),
    ("gonna be out tmrw", "I will be out of the office tomorrow."),
    ("cant make the meeting", "I am unable to attend the meeting."),
    ("pls review asap", "Please review this document at your earliest convenience."),
    ("need this by eod", "I would appreciate receiving this by the end of the day."),
    ("whats the status", "Could you please provide a status update?"),
    ("lets sync later", "Let us schedule time to connect later."),
    ("idk what to do", "I am uncertain about the appropriate next steps."),
    ("fyi the client called", "For your information, the client contacted us."),
    ("sorry for the confusion", "I apologize for any confusion this may have caused."),
    ("can we push the deadline", "Would it be possible to extend the deadline?"),
    ("got it thanks", "Understood. Thank you for the clarification."),
    ("hey team quick update", "Hello team, I would like to share a brief update."),
    ("this looks good to me", "This appears satisfactory from my perspective."),
    ("any updates?", "Do you have any updates you can share?"),
    ("running 10 min late", "I anticipate arriving approximately ten minutes late."),
    ("can u hop on a call", "Would you be available for a brief call?"),
    ("we should talk about this", "I believe we should discuss this matter further."),
    ("not sure i agree", "I am not certain that I agree with that assessment."),
    ("sounds good", "That proposal works for me."),
    ("will do", "I will take care of that."),
    ("my bad", "I take responsibility for that oversight."),
    ("keep me posted", "Please keep me informed of any developments."),
    ("lmk when ur free", "Please let me know when you are available."),
    ("can we reschedule?", "Would it be possible to reschedule our meeting?"),
    ("just following up", "I am following up on my previous message."),
    ("per my last email", "As I noted in my previous email,"),
    ("heads up", "Please be advised that"),
    ("no worries", "There is no issue."),
    ("gotcha", "I understand."),
    ("ping me", "Please contact me when convenient."),
    ("looping in sarah", "I am copying Sarah for visibility."),
    ("out of pocket today", "I will be unavailable today."),
    ("touch base next week", "Let us connect next week."),
    ("draft looks fine", "The draft meets my expectations."),
    ("need more info", "I require additional information."),
    ("works for me", "That time is acceptable to me."),
    ("see u there", "I look forward to seeing you there."),
    ("thanks in advance", "Thank you in advance for your help."),
    ("as discussed", "As we discussed previously,"),
    ("please advise", "Please advise on the recommended course of action."),
    ("for the record", "For the record,"),
    ("please confirm receipt", "Please confirm that you have received this message."),
    ("i have a conflict", "I have a scheduling conflict at that time."),
    ("can you clarify", "Could you please clarify your request?"),
    ("please prioritize this", "Please prioritize this item."),
    ("when you get a chance", "When you have a moment, please"),
    ("appreciate your patience", "Thank you for your patience."),
]
print(len(TRAINING_PAIRS))


In [ ]:
PROMPT_TEMPLATE = "### Input:\n{input}\n\n### Output:\n{output}"


def format_training_text(input_text, output_text):
    return PROMPT_TEMPLATE.format(input=input_text, output=output_text)


def format_inference_prompt(input_text):
    return f"### Input:\n{input_text}\n\n### Output:\n"


def build_dataset(pairs):
    return Dataset.from_dict({"text": [format_training_text(i, o) for i, o in pairs]})


print(build_dataset(TRAINING_PAIRS)[0]["text"])


---
## Part 3: Fine-Tune the Model

### What fine-tuning does

1. Start from pretrained distilgpt2.
2. Show your formatted examples for several **epochs**.
3. Update weights with **gradient descent** to lower **loss** (same core idea as Week 1).
4. Save a **checkpoint** you reload like the original model.

This takes **minutes** on Colab - not the months/dollars of pretraining.

### Hyperparameters (you choose these)

| Setting | Meaning | Start with |
|---------|---------|------------|
| Learning rate | Update step size | `5e-5` and `2e-4` in two runs |
| Epochs | Passes through data | 3 |
| Batch size | Examples per update | 4 (use 2 if OOM) |

Run **two experiments** with different learning rates.

---

## What is Weights & Biases (W&B)?

**Weights & Biases** is an experiment-tracking website used in real ML work.

**Without W&B:** You change learning rate, retrain, forget what you tried, repeat mistakes.

**With W&B:**
- Plots **training loss** over time (line chart)
- Stores **hyperparameters** next to each run
- Lets you **compare runs** side by side

**Setup steps:**
1. Create account: https://wandb.ai
2. Get API key: https://wandb.ai/authorize
3. In notebook run:
   ```python
   import wandb
   wandb.login()
   ```
4. Paste key when prompted.
5. After training, open your project on wandb.ai - submit that **share link**.

If login fails, set `os.environ["WANDB_MODE"] = "disabled"` to train locally (ask instructor about W&B credit).

---

### TODO 2: set hyperparameters. TODO 3: run training cell.


In [ ]:
LEARNING_RATE_RUN1 = 5e-5
LEARNING_RATE_RUN2 = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 4
MAX_LENGTH = 128


In [ ]:
def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)


def train_finetune_run(pairs, learning_rate, output_dir, run_name):
    if learning_rate is None or NUM_EPOCHS is None or BATCH_SIZE is None:
        raise ValueError("Complete TODO 2 first.")
    if len(pairs) < 50:
        raise ValueError(f"Need >= 50 pairs; have {len(pairs)}")

    dataset = build_dataset(pairs).map(tokenize_batch, batched=True, remove_columns=["text"])
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=learning_rate,
        weight_decay=0.01,
        logging_steps=5,
        save_strategy="no",
        report_to="wandb",
        run_name=run_name,
        seed=SEED,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collator)
    start = time.time()
    trainer.train()
    elapsed = time.time() - start
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Saved {output_dir} in {elapsed/60:.1f} min - check W&B for loss curve")
    return model, elapsed


### TODO 3 - Run two training runs

1. Finish TODO 2 (no `None` values).
2. Run `wandb.login()`.
3. Uncomment and run the next cell.
4. Open W&B - you should see **two runs** and two loss curves.
5. Pick the smoother/lower-loss run for Part 4.

**Reading loss:** Should generally decrease. Flat or spiking lines often mean a bad learning rate.


In [ ]:
import os
if os.environ.get('WANDB_MODE') != 'disabled':
    try:
        import wandb
        wandb.login(reinit=True)
    except Exception as exc:
        print('W&B skipped', exc)
        os.environ['WANDB_MODE'] = 'disabled'

finetuned_model_1, time_1 = train_finetune_run(
    TRAINING_PAIRS, LEARNING_RATE_RUN1, './finetuned-run1', f'{TASK_NAME}-lr{LEARNING_RATE_RUN1}')
finetuned_model_2, time_2 = train_finetune_run(
    TRAINING_PAIRS, LEARNING_RATE_RUN2, './finetuned-run2', f'{TASK_NAME}-lr{LEARNING_RATE_RUN2}')
print(f'Run1 {time_1/60:.1f}m Run2 {time_2/60:.1f}m')


---
## Part 4: Evaluate and Reflect

Compare **base** vs **fine-tuned** on prompts **not** in training data.

For fine-tuned runs, use `format_inference_prompt()` so the format matches training.

**Deliverables:**
- W&B project link (2+ runs)
- 3-5 side-by-side comparisons
- One-page reflection (questions in last cell)


In [ ]:
FINETUNED_DIR = "./finetuned-run1"  # switch if run 2 was better

try:
    finetuned_model = AutoModelForCausalLM.from_pretrained(FINETUNED_DIR).to(device)
    finetuned_model.eval()
    print("Loaded", FINETUNED_DIR)
except Exception as e:
    print("Train in Part 3 first.", e)
    finetuned_model = None


In [ ]:
EVAL_PROMPTS = [
    "hey can we move the deadline to friday",
    "pls send the slides when u can",
    "sorry i missed ur call",
    "Question: What is photosynthesis?\nAnswer:",
    "Write a poem about the ocean.",
]

for prompt in EVAL_PROMPTS:
    print("INPUT:", prompt)
    print("\nBASE:")
    print(textwrap.fill(generate_text(base_model, prompt), width=100))
    if finetuned_model:
        print("\nFINE-TUNED:")
        print(textwrap.fill(generate_text(finetuned_model, format_inference_prompt(prompt)), width=100))
    print("=" * 72)


In [ ]:
WANDB_PROJECT_LINK = "https://wandb.ai/your-username/forge-llm-finetune"

part4_reflection = '''
1. Did fine-tuning help? How can you tell?
2. What did W&B loss curves look like?
3. Fine-tune time vs pretraining (Lesson 1)?
4. Tradeoffs: distilgpt2 vs GPT-4 class models?
'''
print("Fill in link + reflection before submitting.")
